# Beam optimization environments: design choices and formulas

This notebook documents the actual configuration shared by `TraceWinEnv` and `SurrogateEnv`: observations, sensitivity, scale calibration, reset noise, actions, trajectory limits, score, reward, and episode termination. All numerical values are read from the current Python configuration.

> This notebook is read-only: it does not run TraceWin and does not change the environment configuration.


In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
from IPython.display import Markdown, display

WORKING_DIR = Path.cwd().resolve()
if (WORKING_DIR / 'beam_optimization').is_dir():
    REPO_ROOT = WORKING_DIR
else:
    PROJECT_ROOT = WORKING_DIR
    while PROJECT_ROOT.name != 'beam_optimization' and PROJECT_ROOT.parent != PROJECT_ROOT:
        PROJECT_ROOT = PROJECT_ROOT.parent
    REPO_ROOT = PROJECT_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from beam_optimization.config.adige import (
    ACTION_SCALE, BAYESIAN_SCALE, BEAM_STATE_DIM, BEAM_STATE_FEATURES,
    DATASET_SCALE, ERROR_SCORE, EXPLORATION_SCALE, FAILURE_PENALTY,
    MAX_STEPS, OBSERVATION_STAGE_MASK, PARAMETERS, RESET_SCALE,
    SCORE_REFERENCES, SCORE_WEIGHTS, STAGE_MARKERS, action_bounds,
    dataset_std_vec, observation_dim, observation_stage_indices,
    observation_stage_labels, reset_std_vec, sensitivity_vec,
)
from beam_optimization.config.offline_utility.scales_calculation import (
    DEFAULT_F_RESET, DEFAULT_K_SIGMA, DEFAULT_K_SIGMA_DATASET,
    compute_scales, verify_constraints,
)

def fmt(value):
    return f'{float(value):.6g}'

def fmt_bound(value):
    return 'unbounded' if value is None else fmt(value)

def table(headers, rows):
    lines = ['| ' + ' | '.join(headers) + ' |', '| ' + ' | '.join(['---'] * len(headers)) + ' |']
    lines.extend('| ' + ' | '.join(str(cell) for cell in row) + ' |' for row in rows)
    return '\n'.join(lines)


## 1. Shared interface and observation space

Both environments inherit the same Gymnasium loop from `BaseBeamEnv`; only the simulator backend changes. Each visible stage contains nine beam features in the configured order. Selected stage matrices are concatenated and flattened:

$$o_t = \operatorname{flatten}(B_t[s_1],\ldots,B_t[s_m]), \qquad \dim(o_t)=m\times BEAM\_STATE\_DIM.$$

The score is not appended to the observation. It is returned separately as `info["score"]`.


In [ ]:
visible_indices = observation_stage_indices()
visible_labels = observation_stage_labels()
action_low, action_high = action_bounds()

stage_rows = []
for index, (marker, visible) in enumerate(zip(STAGE_MARKERS, OBSERVATION_STAGE_MASK)):
    label = 'beam0' if index == 0 else ('final' if index == len(STAGE_MARKERS) - 1 else f'marker_{marker}')
    stage_rows.append([index, marker, f'`{label}`', 'yes' if visible else 'no'])

meanings = {
    'npart_ratio': 'surviving particle fraction',
    'x0': 'horizontal centroid [mm]', 'y0': 'vertical centroid [mm]',
    'SizeX': 'horizontal RMS size [mm]', 'SizeY': 'vertical RMS size [mm]',
    'ex': 'horizontal geometric RMS emittance [mm.mrad]',
    'ey': 'vertical geometric RMS emittance [mm.mrad]',
    "x'0": 'horizontal angular centroid [mrad]',
    "y'0": 'vertical angular centroid [mrad]',
}
feature_rows = [[i, f'`{name}`', meanings[name], 'yes'] for i, name in enumerate(BEAM_STATE_FEATURES)]
interface_rows = [[
    name, f'{len(visible_indices)} × {BEAM_STATE_DIM} = {observation_dim()}',
    str((observation_dim(),)), str(action_low.shape), '`info["score"]`',
] for name in ('TraceWinEnv', 'SurrogateEnv')]

display(Markdown('\n\n'.join([
    table(['Environment', 'Observation formula', 'Obs shape', 'Action shape', 'Score output'], interface_rows),
    '### Beam stages and visibility',
    table(['Stage index', 'TraceWin marker', 'Label', 'Visible to the agent'], stage_rows),
    f'Current observation: **{", ".join(visible_labels)}**, therefore **{observation_dim()} float32 values**.',
    '### Features in each visible stage',
    table(['Index', 'Feature', 'Meaning', 'Used by score'], feature_rows),
])))


## 2. Sensitivity and scale hierarchy

For parameter $p$, `sensitivity_p` is the physical parameter interval estimated to produce an absolute score change of approximately one point near its calibration point:

$$s_p \approx \frac{\Delta p}{|\Delta score|}.$$

Sensitivity is a **local calibration unit**, not a hardware limit and not a guarantee of linear response far from the calibration point. It converts the shared dimensionless scales into parameter-specific physical intervals:

$$\sigma_{dataset,p}=DATASET\_SCALE\,s_p$$
$$\sigma_{reset,p}=RESET\_SCALE\,s_p$$
$$a_{max,p}=ACTION\_SCALE\,s_p$$
$$trajectory_{max,p}=MAX\_STEPS\,a_{max,p}.$$

`EXPLORATION_SCALE` calibrates both dataset sampling and the default Bayesian search width. `RESET_SCALE` and `ACTION_SCALE` are derived offline to keep RL trajectories inside the dataset trust region. Hardware bounds are applied afterwards as clipping limits; they do not define the scales.


In [ ]:
sensitivities = sensitivity_vec()
dataset_std = dataset_std_vec()
reset_std = reset_std_vec()
step_max = action_high.astype(np.float64)
trajectory_max = MAX_STEPS * step_max
calibration = compute_scales(dataset_scale=DATASET_SCALE, max_steps=MAX_STEPS)

k_dataset = DEFAULT_K_SIGMA_DATASET
k_reset = DEFAULT_K_SIGMA
dataset_edge_scale = k_dataset * DATASET_SCALE
reset_excursion_scale = k_reset * RESET_SCALE
trajectory_scale = MAX_STEPS * ACTION_SCALE
combined_scale = reset_excursion_scale + trajectory_scale
constraint_ok = combined_scale <= dataset_edge_scale + 1e-12

scale_rows = [
    ['Exploration scale', '`EXPLORATION_SCALE`', fmt(EXPLORATION_SCALE), 'calibrated from successful TraceWin runs'],
    ['Dataset Gaussian width', '`DATASET_SCALE`', fmt(DATASET_SCALE), 'standard deviation in sensitivity units'],
    ['Bayesian half-width', '`BAYESIAN_SCALE`', fmt(BAYESIAN_SCALE), 'default ± search interval'],
    ['Reset Gaussian width', '`RESET_SCALE`', fmt(RESET_SCALE), 'reset standard deviation'],
    ['Per-step action half-width', '`ACTION_SCALE`', fmt(ACTION_SCALE), 'maximum signed action'],
    ['Episode horizon', '`MAX_STEPS`', str(MAX_STEPS), 'maximum number of actions'],
]
parameter_rows = []
for i, parameter in enumerate(PARAMETERS):
    parameter_rows.append([
        i, f'`{parameter.name}`', fmt(parameter.default), fmt(sensitivities[i]),
        fmt_bound(parameter.hw_min), fmt_bound(parameter.hw_max),
        fmt(dataset_std[i]), fmt(reset_std[i]), f'±{fmt(step_max[i])}', f'±{fmt(trajectory_max[i])}',
    ])

warnings = verify_constraints(
    dataset_scale=DATASET_SCALE, reset_scale=RESET_SCALE,
    action_scale=ACTION_SCALE, max_steps=MAX_STEPS,
)
parts = [
    '### Current global scales',
    table(['Quantity', 'Configuration', 'Value', 'Role'], scale_rows),
    '### Trust-region constraint',
    '```text\nk_sigma × RESET_SCALE + MAX_STEPS × ACTION_SCALE <= k_sigma_dataset × DATASET_SCALE\n```',
    (f'Dataset edge: `{fmt(k_dataset)} × {fmt(DATASET_SCALE)} = {fmt(dataset_edge_scale)}`. '
     f'Worst reset: `{fmt(k_reset)} × {fmt(RESET_SCALE)} = {fmt(reset_excursion_scale)}`. '
     f'Action-only trajectory: `{MAX_STEPS} × {fmt(ACTION_SCALE)} = {fmt(trajectory_scale)}`. '
     f'Combined: **{fmt(combined_scale)}** — constraint **{"satisfied" if constraint_ok else "VIOLATED"}**.'),
    (f'Calibration convention: `k_sigma={fmt(k_reset)}`, `k_sigma_dataset={fmt(k_dataset)}`, '
     f'`f_reset={fmt(DEFAULT_F_RESET)}`. The action-scale ceiling from the trust-region budget is '
     f'`{fmt(calibration["action_scale_max"])}`; the configured value is `{fmt(ACTION_SCALE)}`.'),
]
if warnings:
    parts += ['Hardware-clipping diagnostics:', '```text\n' + '\n'.join(warnings) + '\n```']
parts += [
    '### Per-parameter physical intervals',
    'All intervals come from the sensitivity in the same row. `Trajectory max` is action-only; hardware clipping may reduce the reachable displacement.',
    table(
        ['#', 'Parameter', 'Default', 'Sensitivity', 'HW min', 'HW max', 'Dataset std', 'Reset std', 'Action/step', 'Trajectory max'],
        parameter_rows,
    ),
]
display(Markdown('\n\n'.join(parts)))

labels = ['Dataset trust-region half-width', 'Worst-case reset excursion', 'Action-only full trajectory', 'Reset + full trajectory']
values = [dataset_edge_scale, reset_excursion_scale, trajectory_scale, combined_scale]
colors = ['#4c78a8', '#f2cf5b', '#72b7b2', '#e45756']
fig, ax = plt.subplots(figsize=(9.5, 4.2))
bars = ax.barh(labels, values, color=colors)
ax.axvline(dataset_edge_scale, color='#2f4b7c', linestyle='--', linewidth=1.5, label='dataset edge')
for bar, value in zip(bars, values):
    ax.text(value, bar.get_y() + bar.get_height() / 2, f'  {fmt(value)} × sensitivity', va='center', fontsize=9)
ax.set_xlabel('Dimensionless displacement (multiples of parameter sensitivity)')
ax.set_title('Configured reset and action budget inside the dataset trust region')
ax.grid(axis='x', alpha=0.25)
ax.legend(loc='lower right')
ax.invert_yaxis()
fig.tight_layout()
plt.show()


## 3. Reset, actions, and trajectory

At reset, every parameter starts from its default plus independent Gaussian noise, followed by hardware clipping:

$$\epsilon_p\sim\mathcal{N}(0,(RESET\_SCALE\,s_p)^2), \qquad p_0=\operatorname{clip}_{hardware}(p_{default}+\epsilon_p).$$

`reset(options={"randomize_params": False})` disables this perturbation. The simulator context is then reset: the surrogate samples an input beam and an ensemble member, while TraceWin prepares its real execution context.

Each action is an 18-component vector of **physical parameter deltas**:

$$a_{t,p}=\operatorname{clip}(a^{requested}_{t,p},-ACTION\_SCALE\,s_p,+ACTION\_SCALE\,s_p),$$
$$p_{t+1}=\operatorname{clip}_{hardware}(p_t+a_{t,p}).$$

Actions are cumulative. The action-only displacement satisfies

$$|p_T-p_0|\le T\,ACTION\_SCALE\,s_p\le MAX\_STEPS\,ACTION\_SCALE\,s_p.$$

Including the calibrated worst-case reset gives a separate theoretical allowance:

$$|p_T-p_{default}|\lesssim(k_{reset}RESET\_SCALE+MAX\_STEPS\,ACTION\_SCALE)s_p.$$

Hardware clipping can only reduce these displacements. A successful episode is truncated after `MAX_STEPS`; a failed simulation terminates it immediately.


In [ ]:
w, ref = SCORE_WEIGHTS, SCORE_REFERENCES
score_formula = f"""```text
score = {fmt(w['npart_ratio'])} * clip(npart_ratio, 0, 1)
        - {fmt(w['emittance'])} * ((ex - {fmt(ref['ex'])}) + (ey - {fmt(ref['ey'])}))
        - {fmt(w['offset'])} * ((abs(x0) - {fmt(ref['x0'])}) + (abs(y0) - {fmt(ref['y0'])}))
        - {fmt(w['angle'])} * ((abs(x'0) - {fmt(ref["x'0"])}) + (abs(y'0) - {fmt(ref["y'0"])}))
        - {fmt(w['size'])} * ((SizeX - {fmt(ref['SizeX'])}) + (SizeY - {fmt(ref['SizeY'])}))
```"""
score_rows = [
    ['Transmission', '`npart_ratio`', 'maximize', fmt(w['npart_ratio']), 'clipped to [0, 1]'],
    ['Emittance', '`ex`, `ey`', 'reference-relative', fmt(w['emittance']), f"{fmt(ref['ex'])}, {fmt(ref['ey'])}"],
    ['Centroid offset', '`abs(x0)`, `abs(y0)`', 'minimize magnitude', fmt(w['offset']), f"{fmt(ref['x0'])}, {fmt(ref['y0'])}"],
    ['Angular offset', "`abs(x'0)`, `abs(y'0)`", 'minimize magnitude', fmt(w['angle']), fmt(ref["x'0"]) + ', ' + fmt(ref["y'0"])],
    ['RMS size', '`SizeX`, `SizeY`', 'reference-relative', fmt(w['size']), f"{fmt(ref['SizeX'])}, {fmt(ref['SizeY'])}"],
]
reward_formula = f"""```text
if simulation succeeds:
    reward_t = score_t - score_(t-1)
    terminated = False
    truncated = (step == {MAX_STEPS})
else:
    score_t = {fmt(ERROR_SCORE)}       # reporting sentinel
    reward_t = -{fmt(FAILURE_PENALTY)} # bounded training penalty
    terminated = True
    truncated = False
```"""
display(Markdown('\n\n'.join([
    '## 4. Score, reward, and termination',
    'The score is computed from the **final beam stage** and higher is better. This formula is generated from the current configuration:',
    score_formula,
    table(['Component', 'Features', 'Intent', 'Weight', 'Reference'], score_rows),
    (f'A beam at the references with full transmission scores **{fmt(w["npart_ratio"])}**. '
     'Values better than a reference can receive a linear bonus; the score is not capped above.'),
    '### Reward and episode ending',
    reward_formula,
    (f'`ERROR_SCORE={fmt(ERROR_SCORE)}` is a bookkeeping sentinel, not a physical score, and is not used in the reward difference. '
     f'Failures instead receive `-{fmt(FAILURE_PENALTY)}` and terminate the episode. '
     '`best_score` and `best_params` retain the highest score reached in the episode.'),
])))


## 5. Simulator backends and diagnostics

| Choice | `TraceWinEnv` | `SurrogateEnv` |
| --- | --- | --- |
| Propagation | TraceWin physics simulator | Trained `ModularMLP` ensemble |
| Speed | Slow | Fast |
| Physical failures | Detects TraceWin particle-loss errors | Does not currently classify particle-loss regions |
| Reset context | TraceWin input and execution context | Samples a dataset `beam0` and an ensemble member |
| Gym behavior | Shared `BaseBeamEnv` | Shared `BaseBeamEnv` |

The surrogate predicts continuous beam states and normally reports `success=True` whenever inference succeeds. Failed TraceWin simulations are not beam-state training targets, so parameters that lose all particles in TraceWin may still receive a plausible surrogate prediction. This limitation is not an action-space safety guarantee.

### Render-only diagnostics

`render()` shows parameter trajectories, observed beam features, score/reward, and parameter KNN distance from the reference dataset. KNN distance is a coverage diagnostic: it is **not** part of the observation, score, reward, termination rule, or action clipping. Rendering is lazy and does not affect training steps.

### Design chain at a glance

```text
TraceWin calibration -> sensitivity per parameter
                         +-> DATASET_SCALE -> surrogate training region
                         +-> RESET_SCALE   -> initial episode distribution
                         +-> ACTION_SCALE  -> per-step action bounds -> trajectory bound
beam states -> selected stages -> observation
final beam  -> score -> score difference -> reward
simulation failure -> bounded penalty + terminated=True
```
